# Radar Simulator Tutorial

这个 notebook 面向第一次使用当前 `radar_basics` simulator 的读者，目标是一步一步演示：

1. YAML 配置文件在哪里
2. `load_config(...)` 会把 YAML 解析成什么
3. `simulate_cpi(...)` 会返回什么样的 `raw datacube`
4. 如何用最基础的图把 `(fast-time, slow-time, channel)` 三个维度看清楚
5. `performance` 模式如何反推出一组可仿真的架构参数

这个 tutorial **只停在 raw datacube**，不会继续进入 matched filtering、Doppler FFT 或 angle processing。那部分更适合作为后续单独的 notebook。

## 运行前提

- 已在项目根目录执行过 `uv sync`
- 在 VS Code 里打开这个仓库
- 打开本 notebook 后，选择项目 `.venv` 对应的 Python kernel


In [ ]:
from __future__ import annotations

from pathlib import Path
from pprint import pprint
import math
import tempfile

import matplotlib.pyplot as plt
import numpy as np
import yaml

from radar_basics import SPEED_OF_LIGHT, load_config, resolve_performance_config, simulate_cpi

np.set_printoptions(precision=3, suppress=True)
plt.style.use("default")


## 1. Locate repo assets

VS Code notebook 的当前工作目录有时是仓库根目录，有时是 `notebooks/`。下面这段代码会自动定位 repo root，并找到现成的示例 YAML。


In [ ]:
def find_repo_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        if (candidate / "pyproject.toml").exists() and (candidate / "examples").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository root. Open this notebook from the radar_basics workspace."
    )


REPO_ROOT = find_repo_root()
ARCH_PATH = REPO_ROOT / "examples" / "architecture.yaml"
PERF_PATH = REPO_ROOT / "examples" / "performance.yaml"

print(f"Repo root: {REPO_ROOT}")
print(f"Architecture example: {ARCH_PATH}")
print(f"Performance example:  {PERF_PATH}")


## 2. Architecture YAML walkthrough

先从 `architecture` 模式开始。这个模式要求你直接给出波形、阵列、采样率、脉冲数、发射功率等物理参数；simulator 负责读取这些参数并生成回波。

先把 YAML 原文读出来，再看看 `load_config(...)` 之后数据结构长什么样。


In [ ]:
print(ARCH_PATH.read_text(encoding="utf-8"))

architecture_config = load_config(ARCH_PATH)

print(f"mode: {architecture_config.mode}")
print("\nRadar parameters:")
pprint(architecture_config.radar.__dict__)

print("\nNoise parameters:")
pprint(architecture_config.noise.__dict__)

print("\nTargets:")
for target_index, target in enumerate(architecture_config.scene.targets):
    print(
        f"  target {target_index}: "
        f"range={target.range_m} m, azimuth={target.azimuth_deg} deg, "
        f"radial_velocity={target.radial_velocity_mps} m/s, rcs={target.rcs_m2} m^2"
    )


## 3. First simulation run

现在直接调用 `simulate_cpi(...)`。返回结果里最重要的是：

- `raw_cube`：复数 datacube，shape 是 `(fast_time, slow_time, channel)`
- `axes`：三个轴的坐标
- `truth`：目标真值
- `resolved_config`：最终用于仿真的完整架构参数

下面先定义两个很小的辅助函数：

- `summarize_result(...)`：统一打印结果摘要
- `target_fast_time_index(...)`：根据目标 range 粗略估算它在 fast-time 维上的采样位置


In [ ]:
def summarize_result(result, *, name: str = "result") -> None:
    print(f"{name}.raw_cube.shape: {result.raw_cube.shape}")
    print(f"{name}.raw_cube.dtype:  {result.raw_cube.dtype}")
    print("axes:")
    print(f"  fast_time_s: {len(result.axes.fast_time_s)} samples")
    print(f"  pulse_times_s: {len(result.axes.pulse_times_s)} pulses")
    print(f"  channel_positions_m: {len(result.axes.channel_positions_m)} channels")
    print("truth targets:")
    for target_index, target in enumerate(result.truth):
        print(
            f"  target {target_index}: "
            f"range={target.range_m} m, azimuth={target.azimuth_deg} deg, "
            f"radial_velocity={target.radial_velocity_mps} m/s, rcs={target.rcs_m2} m^2"
        )
    print("resolved_config:")
    for field_name in [
        "range_resolution_m",
        "max_unambiguous_range_m",
        "velocity_resolution_mps",
        "max_unambiguous_velocity_mps",
        "num_elements",
        "num_pulses",
        "sample_rate_hz",
        "pri_s",
    ]:
        value = getattr(result.resolved_config, field_name)
        print(f"  {field_name}: {value}")


def target_fast_time_index(result, target_index: int = 0) -> int:
    target = result.truth[target_index]
    sample_index = math.ceil(
        2.0 * target.range_m * result.resolved_config.sample_rate_hz / SPEED_OF_LIGHT
    )
    return min(sample_index, result.raw_cube.shape[0] - 1)


architecture_result = simulate_cpi(ARCH_PATH)
summarize_result(architecture_result, name="architecture_result")

first_target_fast_time_index = target_fast_time_index(architecture_result, target_index=0)
print(f"\nFirst target is expected near fast-time index {first_target_fast_time_index}.")


## 4. Essential plots for the raw cube

接下来只做三类最基础的图，目的是先把 datacube 的三个维度看清楚：

1. 固定 `(pulse, channel)`，看 fast-time 方向的幅度曲线
2. 固定一个 `channel`，看 `|raw_cube[:, :, channel]|` 的 heatmap
3. 固定 `(fast_time, pulse)`，看 channel 方向的幅度和相位

这些图还不是信号处理结果，它们只是帮助你理解 raw echo 在 tensor 里的组织方式。


In [ ]:
def plot_fast_time_magnitude(result, *, pulse_index: int = 0, channel_index: int = 0, target_index: int = 0) -> None:
    fast_time_us = result.axes.fast_time_s * 1e6
    magnitude_db = 20.0 * np.log10(np.abs(result.raw_cube[:, pulse_index, channel_index]) + 1e-12)
    target_index_sample = target_fast_time_index(result, target_index=target_index)

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(fast_time_us, magnitude_db, color="tab:blue")
    ax.axvline(
        fast_time_us[target_index_sample],
        color="tab:red",
        linestyle="--",
        label=f"target {target_index} expected delay",
    )
    ax.set_title(f"Fast-time magnitude at pulse={pulse_index}, channel={channel_index}")
    ax.set_xlabel("fast-time (us)")
    ax.set_ylabel("magnitude (dB)")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_raw_cube_heatmap(result, *, channel_index: int = 0) -> None:
    magnitude_db = 20.0 * np.log10(np.abs(result.raw_cube[:, :, channel_index]) + 1e-12)
    pulse_times_ms = result.axes.pulse_times_s * 1e3
    fast_time_us = result.axes.fast_time_s * 1e6
    x_max = pulse_times_ms[-1] if pulse_times_ms.size > 1 else pulse_times_ms[0] + result.resolved_config.pri_s * 1e3

    fig, ax = plt.subplots(figsize=(8, 4))
    image = ax.imshow(
        magnitude_db,
        aspect="auto",
        origin="lower",
        extent=[pulse_times_ms[0], x_max, fast_time_us[0], fast_time_us[-1]],
        cmap="viridis",
    )
    ax.set_title(f"|raw_cube[:, :, {channel_index}]| heatmap")
    ax.set_xlabel("slow-time / pulse time (ms)")
    ax.set_ylabel("fast-time (us)")
    fig.colorbar(image, ax=ax, label="magnitude (dB)")
    plt.tight_layout()
    plt.show()


def plot_channel_snapshot(result, *, fast_time_index: int, pulse_index: int = 0, title_prefix: str = "") -> None:
    snapshot = result.raw_cube[fast_time_index, pulse_index, :]
    channels = np.arange(result.raw_cube.shape[2])

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(channels, np.abs(snapshot), marker="o")
    axes[0].set_title(f"{title_prefix}channel magnitude")
    axes[0].set_xlabel("channel index")
    axes[0].set_ylabel("|value|")
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(channels, np.unwrap(np.angle(snapshot)), marker="o")
    axes[1].set_title(f"{title_prefix}channel phase")
    axes[1].set_xlabel("channel index")
    axes[1].set_ylabel("phase (rad)")
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


plot_fast_time_magnitude(architecture_result, pulse_index=0, channel_index=0, target_index=0)
plot_raw_cube_heatmap(architecture_result, channel_index=0)
plot_channel_snapshot(
    architecture_result,
    fast_time_index=first_target_fast_time_index,
    pulse_index=0,
    title_prefix="First target ",
)


## 5. One edit experiment

现在做一次最小实验：

- 读取 `examples/architecture.yaml`
- 只改第一个目标的 `azimuth_deg`
- 把改后的 YAML 写到临时文件
- 再跑一次 `simulate_cpi(...)`

这样可以保持“**用户通过 YAML 驱动 simulator**”这条主线不变，同时又能直观看到角度变化如何影响 channel 维的相位和波束衰减。


In [ ]:
with ARCH_PATH.open("r", encoding="utf-8") as handle:
    edited_yaml = yaml.safe_load(handle)

original_azimuth_deg = edited_yaml["scene"]["targets"][0]["azimuth_deg"]
edited_yaml["scene"]["targets"][0]["azimuth_deg"] = 30.0

with tempfile.TemporaryDirectory() as tmpdir:
    edited_path = Path(tmpdir) / "edited_architecture.yaml"
    edited_path.write_text(yaml.safe_dump(edited_yaml, sort_keys=False), encoding="utf-8")
    edited_result = simulate_cpi(edited_path)

edited_azimuth_deg = edited_yaml["scene"]["targets"][0]["azimuth_deg"]
original_snapshot = architecture_result.raw_cube[first_target_fast_time_index, 0, :]
edited_snapshot = edited_result.raw_cube[target_fast_time_index(edited_result, 0), 0, :]

def transmit_field_gain(result, azimuth_deg: float) -> float:
    steering_term = math.sin(math.radians(result.resolved_config.steering_azimuth_deg))
    azimuth_term = math.sin(math.radians(azimuth_deg))
    return float(
        np.abs(
            np.mean(
                np.exp(
                    1j
                    * 2.0
                    * math.pi
                    * result.axes.channel_positions_m
                    * (azimuth_term - steering_term)
                    / result.resolved_config.wavelength_m
                )
            )
        )
    )

original_gain = transmit_field_gain(architecture_result, original_azimuth_deg)
edited_gain = transmit_field_gain(edited_result, edited_azimuth_deg)

print(f"First target azimuth: {original_azimuth_deg} deg -> {edited_azimuth_deg} deg")
print(f"Predicted transmit field gain before edit: {original_gain:.3f}")
print(f"Predicted transmit field gain after edit:  {edited_gain:.3f}")
print("Note: the raw snapshot still contains AWGN, so the plotted magnitude change will be less dramatic than the ideal beam gain change.")

channels = np.arange(architecture_result.raw_cube.shape[2])
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].plot(channels, np.abs(original_snapshot), marker="o", label=f"{original_azimuth_deg} deg")
axes[0].plot(channels, np.abs(edited_snapshot), marker="o", label=f"{edited_azimuth_deg} deg")
axes[0].set_title("Channel magnitude comparison")
axes[0].set_xlabel("channel index")
axes[0].set_ylabel("|value|")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(channels, np.unwrap(np.angle(original_snapshot)), marker="o", label=f"{original_azimuth_deg} deg")
axes[1].plot(channels, np.unwrap(np.angle(edited_snapshot)), marker="o", label=f"{edited_azimuth_deg} deg")
axes[1].set_title("Channel phase comparison")
axes[1].set_xlabel("channel index")
axes[1].set_ylabel("phase (rad)")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()


## 6. Performance-mode walkthrough

`performance` 模式和 `architecture` 模式的区别是：

- `architecture`：你直接给出物理参数
- `performance`：你先给出希望达到的性能目标，simulator 再按当前实现里的闭式公式反推出一组架构参数

下面先读 YAML，再显式调用 `resolve_performance_config(...)` 看看它反推出了什么。


In [ ]:
print(PERF_PATH.read_text(encoding="utf-8"))

performance_config = load_config(PERF_PATH)
resolved_performance = resolve_performance_config(performance_config)

print(f"mode: {performance_config.mode}")
print("\nPerformance-mode input:")
pprint(performance_config.radar.__dict__)

print("\nDerived architecture parameters:")
for field_name in [
    "chirp_bandwidth_hz",
    "pri_s",
    "num_pulses",
    "num_elements",
    "sample_rate_hz",
    "peak_tx_power_w",
]:
    value = getattr(resolved_performance, field_name)
    print(f"  {field_name}: {value}")


In [ ]:
performance_result = simulate_cpi(PERF_PATH)
summarize_result(performance_result, name="performance_result")

print("\nShape comparison:")
print(f"  architecture example: {architecture_result.raw_cube.shape}")
print(f"  performance example:  {performance_result.raw_cube.shape}")


## 7. Wrap-up

到这里你已经走完了当前 simulator 的最基本使用路径：

- 用 `architecture` 模式时，你直接控制雷达的波形、阵列和采样参数
- 用 `performance` 模式时，你先给性能目标，再让 simulator 反推出一组可运行的架构参数
- `simulate_cpi(...)` 的核心输出是一个复数 `raw datacube`，shape 是 `(fast_time, slow_time, channel)`
- `fast_time` 主要承载距离相关信息，`slow_time` 主要承载多普勒相关信息，`channel` 主要承载阵列角度相关信息

下一步如果要继续学习 radar signal processing，最自然的延伸是新开一个 notebook，围绕这个 `raw_cube` 继续做：

- matched filtering / pulse compression
- Doppler processing
- angle processing / beamforming

但那会是下一阶段；当前这个 tutorial 的目标，就是先把 simulator 本身用明白。
